In [25]:
import os
import sys
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# --- 0. REPRODUCIBILITY ---
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- 1. PATHS & IMPORTS ---
root_dir = os.path.abspath("..")
for path in [root_dir,
             os.path.abspath("../current_setpoints"),
             os.path.abspath("../current_setpoints/optimization"),
             os.path.abspath("../current_setpoints/model")]:
    if path not in sys.path:
        sys.path.append(path)

from current_setpoints.data import FluxValues, IEEEMachine2
from current_setpoints.utils import (
    NeuralTorquePredictor, evaluate_model, load_aggregated_csv_data,
)
from current_setpoints.utils.neural import torq_analytical   # adjust path if needed

AGGREGATED_FILE_PATH = '../data/aggregated_file_means.csv'
GRID_PICKLE_PATH     = '../data/compensated_grid.pkl'   # <-- adjust
COLUMN_MAP = {'omega':'omega','id1':'id1','iq1':'iq1','id3':'id3','iq3':'iq3','torq':'torq'}
HIDDEN_SIZE      = 12
FINAL_BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. EXPERIMENTAL DATA (identical splits to the original trainer) ---
flux_values = FluxValues()
machine = IEEEMachine2(flux_values)
data = load_aggregated_csv_data(AGGREGATED_FILE_PATH, COLUMN_MAP)
X = data[['omega','id1','iq1','id3','iq3']].values.astype(np.float32)
y = data[['torq']].values.astype(np.float32)

B_list = []
for row in X:
    machine.update_state(omega=row[0], vec_curr_dq=row[1:])
    B_list.append(machine.vec_b.copy())
B_array = np.array(B_list, dtype=np.float32)

X_train_val, X_test, y_train_val, y_test, B_train_val, B_test = train_test_split(
    X, y, B_array, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val, B_train, B_val = train_test_split(
    X_train_val, y_train_val, B_train_val, test_size=0.20, random_state=42)

scaler_X = StandardScaler()
scaler_X.fit(X_train)
X_test_norm  = scaler_X.transform(X_test).astype(np.float32)

test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test_norm).float(),
                  torch.from_numpy(y_test).float(),
                  torch.from_numpy(B_test).float()),
    batch_size=FINAL_BATCH_SIZE, shuffle=False)

# --- 3. STATIC b ---
_, ieee_flux_torq = flux_values.get_flux("IEEEMachine2", 0.0, np.zeros(4))
B_STATIC = (machine.n_phases * machine.n_ppairs / 4) * (machine.mat_crossc @ ieee_flux_torq)
B_STATIC_TENSOR = torch.from_numpy(B_STATIC).float().to(DEVICE)

# --- 4. TEACHER (old buggy analytical + NN_old residual) ---
class TeacherModel(NeuralTorquePredictor):
    """out_old = i^T A i + b^T i + NN_old(x)        (no factor of 2)"""
    def forward(self, x_normed, _B_dyn):
        x_phys = x_normed * self.x_std + self.x_mean
        i_dq = x_phys[:, 1:]
        t_quad = torch.einsum("bi,ij,bj->b", i_dq, self.A_TENSOR, i_dq)
        t_lin  = i_dq @ B_STATIC_TENSOR
        t_ana  = (t_quad + t_lin).unsqueeze(1)
        t_res  = self.fc2(self.gelu(self.fc1(x_normed)))
        return t_ana + t_res

teacher = TeacherModel(5, HIDDEN_SIZE, scaler_X, machine, DEVICE).to(DEVICE)
state_dict = torch.load('../weights/NTM_Best_Model_HALF.pth', map_location=DEVICE)
state_dict.pop("B_TENSOR", None)
teacher.load_state_dict(state_dict, strict=False)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

# --- 5. STUDENT WITH LINEAR BYPASS ---
class StudentWithBypass(NeuralTorquePredictor):
    """
    Architecture: analytical baseline + nonlinear neural residual
                  + learnable affine correction on the current vector.

    The affine bypass exists so any linear-in-currents component of the
    residual is represented exactly through a dedicated linear path,
    rather than approximated through GELU activations.  This keeps the
    nonlinear residual small and well-behaved, and makes the model robust
    to changes in how the analytical baseline is parameterized.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        n_currents = self.x_mean.shape[0] - 1
        self.linear_bypass = nn.Linear(n_currents, 1, bias=False)

    def forward(self, x_normed, B_tensor):
        x_phys = x_normed * self.x_std + self.x_mean
        i_dq = x_phys[:, 1:]
        t_ana = torq_analytical(i_dq, self.A_TENSOR, B_tensor)
        t_res = self.fc2(self.gelu(self.fc1(x_normed)))
        t_byp = self.linear_bypass(i_dq)
        return t_ana + t_res + t_byp

student = StudentWithBypass(5, HIDDEN_SIZE, scaler_X, machine, DEVICE).to(DEVICE)
student.load_state_dict(state_dict, strict=False)   # warm-start fc1, fc2

# Initialize bypass to the EXACT correction:
#   teacher  = ana_quad +     b^T i + NN_old(x)
#   student  = ana_quad + 2 b^T i + NN_old(x) + w_byp^T i
#   match  =>  w_byp = -b
with torch.no_grad():
    student.linear_bypass.weight.copy_(-B_STATIC_TENSOR.view(1, -1))

student.eval()

# --- 6. VERIFY ON GRID ---
print("Loading optimization grid for verification...")
with open(GRID_PICKLE_PATH, 'rb') as f:
    grid = pickle.load(f)

om_vec = grid['om_vec']; T_vec = grid['T']
isd1 = grid['isd1']; isq1 = grid['isq1']
isd3 = grid['isd3']; isq3 = grid['isq3']
max_T = grid['max_T'].squeeze()

TT, OM = np.meshgrid(T_vec, om_vec, indexing='ij')
omega_arr    = OM.ravel()
target_arr   = TT.ravel()
currents_arr = np.column_stack([isd1.ravel(), isq1.ravel(),
                                isd3.ravel(), isq3.ravel()])

finite_mask   = np.isfinite(currents_arr).all(axis=1) & np.isfinite(target_arr)
max_T_grid    = np.broadcast_to(max_T[None, :], OM.shape).ravel()
envelope_mask = target_arr <= max_T_grid + 1e-6
prelim_valid  = finite_mask & envelope_mask

X_prelim = np.column_stack([omega_arr[prelim_valid], currents_arr[prelim_valid]]).astype(np.float32)
X_prelim_norm = scaler_X.transform(X_prelim).astype(np.float32)
B_prelim = np.tile(B_STATIC.astype(np.float32), (len(X_prelim), 1))

with torch.no_grad():
    chunk = 8192
    s_parts, t_parts = [], []
    for i in range(0, len(X_prelim_norm), chunk):
        x_t = torch.from_numpy(X_prelim_norm[i:i+chunk]).to(DEVICE)
        b_t = torch.from_numpy(B_prelim[i:i+chunk]).to(DEVICE).unsqueeze(-1)
        s_parts.append(student(x_t, b_t).cpu().numpy())
        t_parts.append(teacher(x_t, b_t).cpu().numpy())
    y_student = np.concatenate(s_parts).squeeze()
    y_teacher = np.concatenate(t_parts).squeeze()

target_prelim = target_arr[prelim_valid]
sanity_err    = np.abs(y_teacher - target_prelim)
consistent    = sanity_err < 0.1
grid_diffs    = np.abs(y_student - y_teacher)[consistent]

print(f"\nVerification on {consistent.sum()} grid cells:")
print(f"  Grid mean |student - teacher| : {grid_diffs.mean():.3e} Nm")
print(f"  Grid max  |student - teacher| : {grid_diffs.max():.3e} Nm")
print(f"  (pure float32 multiplication noise; the bypass is exact by construction)")

# --- 7. ALSO REPORT PERFORMANCE ON HELD-OUT DATA ---
criterion = nn.MSELoss()
final_test_rmse = evaluate_model(student, test_loader, criterion, DEVICE)
with torch.no_grad():
    test_diffs = []
    for x, _, b in test_loader:
        x = x.to(DEVICE); b = b.to(DEVICE).unsqueeze(-1)
        test_diffs.append((student(x, b) - teacher(x, b)).abs().cpu().numpy())
    test_diffs = np.concatenate(test_diffs)

print(f"\nTest set ({len(X_test_norm)} held-out samples):")
print(f"  Test RMSE vs experimental data : {final_test_rmse:.6f} Nm")
print(f"  Test max  |student - teacher|  : {test_diffs.max():.3e} Nm")

# --- 8. SAVE ---
torch.save(student.state_dict(), '../weights/NTM_Cloned_Corrected.pth')
np.save('../weights/NTM_Cloned_Corrected_Scaler.npy',
        np.array({"mean": scaler_X.mean_, "scale": scaler_X.scale_}, dtype=object),
        allow_pickle=True)
print("\nCloned model + scaler saved.")

Loaded 174 valid data points from CSV.
Loading optimization grid for verification...

Verification on 35213 grid cells:
  Grid mean |student - teacher| : 1.109e-07 Nm
  Grid max  |student - teacher| : 9.537e-07 Nm
  (pure float32 multiplication noise; the bypass is exact by construction)

Test set (27 held-out samples):
  Test RMSE vs experimental data : 0.048554 Nm
  Test max  |student - teacher|  : 4.768e-07 Nm

Cloned model + scaler saved.
